# Loading data

In [3]:
from assaybench.dataset.dataset import AssayBenchDataset

In [4]:
dataset_path = "/cv/data/braid/gnesys/datasets/screensQA/biogrid_v0.4_combined"
split_type = "year"  # or "random"
fold = 0  # which fold to use in the given split type

ds = AssayBenchDataset(
                dataset_path=dataset_path,
                split_type=split_type,
                fold=fold,
            )


train,val,test = ds.get_train_test_split()
print(f"Number of screens in train: {len(train)}")
print(f"Number of screens in val: {len(val)}")
print(f"Number of screens in test: {len(test)}")

Number of screens in train: 1349
Number of screens in val: 218
Number of screens in test: 334


# Scoring model

In [5]:
from assaybench.benchmark.metrics import RankingMetrics

In [7]:
top_list = [ex["answer"] for ex in train]
# select top 100 most common answers
from collections import Counter
counter = Counter(top_list)
most_common = counter.most_common(100)
most_common_answers = [x[0] for x in most_common]
print(f"Most common answers: {most_common_answers}")

def top_100_answers(prompt):
    return most_common_answers

metric_fn = RankingMetrics(k_values=[10,100])

Most common answers: ['CUL2, KLHDC3, PRAMEF7, PRAMEF6, PRAMEF9, PRAMEF8, PRAMEF4, PRAMEF5, PRAMEF22, PRAMEF26', 'SLC25A1, SARS2, COX10, PPWD1, DHODH, CCT7, TAFAZZIN, KARS1, SHMT2, MRPS15', 'FDX1, GLG1, GLI2, GLI3, GLI4, GLIPR1, GLIPR1L1, GLIPR1L2, GLIPR2, GLIS1', 'ROCK2, PAWR, BANF2, RHOA, ZNF587B, GNAZ, DYNAP, IFNGR2, GSK3B, TC2N', 'CAB39, PDCD10, NF2, AMOTL2, FRYL, TAOK1, SPOP, IFNA13, LATS2, WASHC2C', 'PDCD10, CAB39, NF2, FRYL, AMOTL2, MIR1299, WASHC2C, IFNA13, MIR3135B, SPOP', 'TRPC4AP, DDB1, WDR76, WDTC1, WDR59, WDR53, WDR5, WDR26, WDR24, WDR12', 'PTPMT1, CIAO2B, TUBG1, RPL11, EIF1AX, HNRNPA1, IRAK1, COQ2, ATP5ME, CAPZB', 'TOP1, HGFAC, HGH1, HGS, HGSNAT, HHAT, HHATL, HHEX, HHIP, HHIPL1', 'TOP2A, CDK6, TAF3, MYB, RPL36, HDGFL3, HDLBP, HDX, HEATR1, HEATR9', 'TOP2A, CDK6, ANKRD17, HDLBP, HDX, HEATR1, HEATR9, HECTD1, HECTD2, HECW1', 'POLR1D, ATF5, CENPI, SEM1, POLR2H, LSM6, WDR5, HJURP, PDCD11, ZC3H8', 'CEBPA, TGFBR2, CDKN1C, PLK4, PRTG, TP73, SPI1, RHOXF2B, RHOXF2, FOXL2NB', 'WDR43, 

In [8]:
metrics = {ex["dataset_name"]: metric_fn.evaluate(ex["question"],
                                                  ex["relevance_genes"],
                                                  ex["relevance_scores"]) for ex in val}

In [14]:
adncg_at_100 = [m["adjusted_ndcg@100"] for m in metrics.values()]
print(f"Average adjusted nDCG@100: {sum(adncg_at_100)/len(adncg_at_100)}")

Average adjusted nDCG@100: 0.011943275750583024
